[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q4/02_classifier.ipynb)

# R1-Q4 Week 3 — Train the stress classifier

### This notebook produces `classifier.pkl`, the trained model the transfer test runs on next week.

This notebook trains a multi-class stress classifier on the integrated AtGenExpress training set produced in Week 2. The trained classifier is the input to the cross-dataset transfer test in Week 4, and the within-AtGenExpress performance numbers will go into your Week 3 draft presentation.

By the end of this notebook you will have:

- A trained multi-class stress classifier on integrated AtGenExpress (`classifier.pkl`), ready for the cross-dataset transfer test in Week 4.
- The held-out AtGenExpress test rows (`atgenexpress_test_split.parquet`), parallel in shape to a chunk of `integrated_matrix.parquet`. The Week 4 transfer test reuses this so the cross-dataset comparison runs on the same samples evaluated here.
- Within-AtGenExpress held-out performance metrics (overall accuracy, per-class precision and recall) saved as `classifier_metrics.parquet`.
- The confusion matrix on the held-out AtGenExpress test set (`classifier_confusion.parquet`), as a labeled square table of true class × predicted class counts. Notebook 03 compares this against the cross-dataset confusion matrix to see whether the same class confusions transfer.
- A `classifier_summary.json` recording the model architecture, training hyperparameters, train/test split sizes, and within-AtGenExpress performance, ready for use in your Week 3 draft presentation.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R1-Q4 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q4")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Defensive load of the three required inputs and cross-file consistency checks

Notebook 01 wrote `integrated_matrix.parquet` and `integration_summary.json` to `OUTPUT_DIR`; Notebook 00 wrote `precommit.json` to the same location. This notebook reads all three before any training begins.

The work splits across two code cells:

1. **Load** — each file from `OUTPUT_DIR`, with structure validation in isolation.
2. **Cross-check** — the three loaded objects have to agree (`wang_168h_handling` decision matches the `ood` column population, expected metadata columns present, no missing gene values, and so on).

If any file is missing, malformed, or inconsistent with the others, the load raises an error and points you back to the notebook that produced the offender. Re-run that notebook's closeout, then re-run from here.

One additional refusal fires at the end of the cross-check: if your Notebook 01 integration gate verdict was `"fail"`, training does not run. NB02 only continues on `"pass"` or `"partial"` — and on `"partial"`, the rationale prints prominently so you remember what you're carrying forward.

In [ ]:
import json
import pandas as pd

PRECOMMIT_PATH           = OUTPUT_DIR / "precommit.json"
INTEGRATED_PATH          = OUTPUT_DIR / "integrated_matrix.parquet"
INTEGRATION_SUMMARY_PATH = OUTPUT_DIR / "integration_summary.json"


# ----- precommit.json -----
# Same expected structure as Notebook 01 — three top-level keys with
# primary-field enums. Validates that Notebook 00's closeout actually
# wrote the file we need.

EXPECTED_PRECOMMIT = {
    "data_source_and_scope": ("source",   {"GEO", "TAIR_NASCArrays"}),
    "wang_168h_handling":    ("decision", {"exclude", "include_as_ood"}),
    "above_chance_null":     ("decision", {"binary_cold_vs_control", "multiclass_behavior"}),
}

if not PRECOMMIT_PATH.exists():
    raise FileNotFoundError(
        f"{PRECOMMIT_PATH} not found. "
        "Run Notebook 00 to its closeout cell, then re-run this cell."
    )

with PRECOMMIT_PATH.open() as f:
    precommit = json.load(f)

for top_key, (field, allowed) in EXPECTED_PRECOMMIT.items():
    if top_key not in precommit:
        raise KeyError(
            f"precommit.json is missing top-level key '{top_key}'. "
            "Re-run Notebook 00's closeout."
        )
    value = precommit[top_key].get(field)
    if value not in allowed:
        raise ValueError(
            f"precommit['{top_key}']['{field}'] = {value!r}, "
            f"but expected one of {sorted(allowed)}. "
            "Fix the relevant Precommit section in Notebook 00 and re-run its closeout."
        )


# ----- integrated_matrix.parquet -----
# Wide table from Notebook 01 Section 9. Samples are rows; columns are
# gene values (on the ComBat-corrected scale) plus three metadata
# columns: batch, stress_label, ood.

if not INTEGRATED_PATH.exists():
    raise FileNotFoundError(
        f"{INTEGRATED_PATH} not found. "
        "Run Notebook 01 to its closeout cell, then re-run this cell."
    )

integrated_matrix = pd.read_parquet(INTEGRATED_PATH)

REQUIRED_META_COLS = {"batch", "stress_label", "ood"}
missing_meta = REQUIRED_META_COLS - set(integrated_matrix.columns)
if missing_meta:
    raise KeyError(
        f"integrated_matrix.parquet is missing metadata column(s): {sorted(missing_meta)}. "
        "Re-run Notebook 01 — Section 9 assembles these columns."
    )


# ----- integration_summary.json -----
# Pattern D verdict from Notebook 01 Section 8. Per-part verdicts
# (pass/partial/fail) plus rationales, with an "overall" rollup.

if not INTEGRATION_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"{INTEGRATION_SUMMARY_PATH} not found. "
        "Run Notebook 01 to its closeout cell, then re-run this cell."
    )

with INTEGRATION_SUMMARY_PATH.open() as f:
    integration_summary = json.load(f)

if "overall" not in integration_summary or "verdict" not in integration_summary["overall"]:
    raise KeyError(
        "integration_summary.json is missing 'overall.verdict'. "
        "Re-run Notebook 01's Section 8 gate cell."
    )

ALLOWED_VERDICTS = {"pass", "partial", "fail"}
verdict = integration_summary["overall"]["verdict"]
if verdict not in ALLOWED_VERDICTS:
    raise ValueError(
        f"integration_summary['overall']['verdict'] = {verdict!r}, "
        f"but expected one of {sorted(ALLOWED_VERDICTS)}."
    )


# ----- Summary printout -----
print("Loaded three files from OUTPUT_DIR.")
print()
print("precommit.json")
for top_key, (field, _) in EXPECTED_PRECOMMIT.items():
    print(f"  {top_key}.{field} = {precommit[top_key][field]!r}")
print()
print("integrated_matrix.parquet")
print(f"  shape:           {integrated_matrix.shape[0]:,} samples x {integrated_matrix.shape[1]:,} columns")
print(f"  batch breakdown: {integrated_matrix['batch'].value_counts().to_dict()}")
print()
print("integration_summary.json")
print(f"  overall verdict: {verdict!r}")

**What's now in memory:**

- `precommit` — three-decision dict from Notebook 00.
- `integrated_matrix` — samples × (genes + `batch` + `stress_label` + `ood`).
- `integration_summary` — verdict dict from Notebook 01's integration gate.

Each file passes its own structure check. The next cell verifies the three fit together — that the `ood` column in `integrated_matrix` matches your `wang_168h_handling` decision in `precommit`, that gene columns are fully populated, and that the gate verdict isn't `"fail"`. If any cross-check fails, the cell raises and names the likely culprit.

In [ ]:
# ----- integrated_matrix structural checks -----

if not integrated_matrix.index.is_unique:
    n_dupes = integrated_matrix.index.duplicated().sum()
    raise ValueError(
        f"integrated_matrix has {n_dupes:,} duplicate sample ID(s). "
        "Re-run Notebook 01 — Section 9's concat should not produce duplicates."
    )

if integrated_matrix["ood"].dtype != bool:
    raise TypeError(
        f"integrated_matrix['ood'] dtype is {integrated_matrix['ood'].dtype}, expected bool. "
        "Re-run Notebook 01's Section 9 — the ood column should be assembled as bool."
    )

gene_cols = integrated_matrix.columns.difference(["batch", "stress_label", "ood"])
n_nans = integrated_matrix[gene_cols].isna().sum().sum()
if n_nans > 0:
    raise ValueError(
        f"integrated_matrix gene columns contain {n_nans:,} NaN value(s). "
        "Re-run Notebook 01 — Section 5's gene intersection should leave no missing values."
    )

unexpected_batches = set(integrated_matrix["batch"].unique()) - {"atgenexpress", "wang"}
if unexpected_batches:
    raise ValueError(
        f"integrated_matrix['batch'] contains unexpected values: {sorted(unexpected_batches)}. "
        "Re-run Notebook 01 — Section 9 assigns batch labels."
    )


# ----- precommit <-> integrated_matrix consistency -----
# The wang_168h_handling decision determines the shape of the ood column.
# Drift between the two would mean Notebook 01 ran on a different precommit
# than the one currently on disk.

handling = precommit["wang_168h_handling"]["decision"]
wang_mask = integrated_matrix["batch"] == "wang"
atgen_mask = integrated_matrix["batch"] == "atgenexpress"
n_wang_ood  = integrated_matrix.loc[wang_mask,  "ood"].sum()
n_atgen_ood = integrated_matrix.loc[atgen_mask, "ood"].sum()

if n_atgen_ood > 0:
    raise ValueError(
        f"AtGenExpress rows should never carry ood=True, but {n_atgen_ood} do. "
        "Re-run Notebook 01 — Section 9 assigns False to all AtGenExpress rows."
    )

if handling == "exclude" and n_wang_ood > 0:
    raise ValueError(
        f"precommit.wang_168h_handling = 'exclude', but {n_wang_ood} Wang row(s) "
        "carry ood=True. The two files disagree — most likely Notebook 01 ran on a "
        "different precommit and was never re-run after the decision changed. "
        "Re-run Notebook 01 from the top."
    )

if handling == "include_as_ood" and n_wang_ood == 0:
    raise ValueError(
        "precommit.wang_168h_handling = 'include_as_ood', but no Wang rows carry "
        "ood=True. The two files disagree — re-run Notebook 01 from the top."
    )


# ----- Gate verdict refuse-on-fail backstop -----
# Notebook 01's Section 8 already raises on a 'fail' verdict before writing
# integration_summary.json, so reaching this cell with verdict == 'fail'
# would mean the file was hand-edited or copied from elsewhere. Belt-and-
# suspenders refuses anyway.

if verdict == "fail":
    raise RuntimeError(
        "integration_summary.json reports overall verdict = 'fail'. "
        "Notebook 02 will not run. Return to Notebook 01, fix what failed, "
        "re-run the integration gate, then come back here."
    )

if verdict == "partial":
    rationale = integration_summary["overall"].get("rationale", "(no rationale recorded)")
    print("=" * 60)
    print("WARNING: integration gate verdict was 'partial'.")
    print()
    print("Rationale from Notebook 01:")
    print(f"  {rationale}")
    print()
    print("Training will proceed, but make sure you understand the")
    print("trade-off you're carrying forward. Document it in EQ#3.")
    print("=" * 60)
else:
    print(f"Integration gate verdict: {verdict!r}. Cleared.")

print()
print("All cross-checks passed.")

**What just happened:** `precommit`, `integrated_matrix`, and `integration_summary` are all loaded and verified to agree with each other. The three are now safe to use throughout the notebook.

A `"partial"` gate verdict prints a prominent warning but does not stop training. NB02 trusts your Notebook 01 judgment — if you marked the integration as marginal-but-acceptable, training proceeds. The constraint is on you to document the compromise in EQ#3.

Section 2 splits the AtGenExpress portion of `integrated_matrix` (rows where `batch == "atgenexpress"`) into train and held-out test sets. The Wang portion is ignored by Notebook 02 — Notebook 03 loads `integrated_matrix` separately to access it for the cross-dataset transfer evaluation.

### 2) Split the AtGenExpress portion of the integrated matrix into train and held-out test


Notebook 02 trains and evaluates *within* AtGenExpress — Wang 2023 is reserved for Notebook 03's cross-dataset transfer test. So the first thing this section does is take the rows where `batch == "atgenexpress"`, then split them into a training set and a held-out test set.

**The split is stratified by `stress_label`.** This matters because the AtGenExpress stress classes are imbalanced — cold, salt, drought, heat, and the others were not sampled in equal numbers. A naive random 75/25 split would, by chance, leave some held-out classes with very few samples (or zero) and skew the per-class precision and recall numbers Section 4 computes. Stratifying forces each class to be split 75/25 *internally*, so the train and test sets preserve the same class composition as the full AtGenExpress pool.

**75/25, not 80/20.** Standard practice in machine learning is 80/20. The extra five percentage points on the test side here are a deliberate trade-off in favor of the smaller stress classes — at 80/20 some of them would have only two or three samples in the held-out test set, which is too few to compute a meaningful per-class recall. 75/25 buys those classes a slightly larger held-out N at the cost of slightly less training data. Note this trade-off in your Week 3 reflection: the "right" ratio is dataset-dependent, and here the binding constraint is the smallest class, not the largest.

**Random seed is fixed.** Reproducibility — the same `random_state` here means you (or anyone re-running this notebook) gets the same train/test split, and therefore the same downstream numbers.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42  # Fixed for reproducibility.
TEST_FRACTION = 0.25

# Take the AtGenExpress portion only. Notebook 03 will load the integrated
# matrix separately to access the Wang rows for the transfer test.
atgen_mask = integrated_matrix["batch"] == "atgenexpress"
atgen_df = integrated_matrix.loc[atgen_mask].copy()

# Separate features (gene expression values) from labels.
# The three metadata columns — batch, stress_label, ood — are not features.
metadata_cols = ["batch", "stress_label", "ood"]
gene_cols = atgen_df.columns.difference(metadata_cols)

X = atgen_df[gene_cols]
y = atgen_df["stress_label"]

# Stratified split. stratify=y forces each stress class to be divided 75/25
# internally, preserving the class composition in both train and test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_FRACTION,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Report split sizes and per-class breakdown so the imbalance is visible
# before training, not surprising during evaluation.
print(f"AtGenExpress total:   {len(atgen_df):,} samples")
print(f"  Train: {len(X_train):,} ({len(X_train) / len(atgen_df):.0%})")
print(f"  Test:  {len(X_test):,}  ({len(X_test)  / len(atgen_df):.0%})")
print()

class_breakdown = pd.DataFrame({
    "train_n": y_train.value_counts(),
    "test_n":  y_test.value_counts(),
}).fillna(0).astype(int)
class_breakdown["total_n"] = class_breakdown["train_n"] + class_breakdown["test_n"]
class_breakdown = class_breakdown.sort_values("total_n", ascending=False)

print("Per-class breakdown (sorted by total N):")
print(class_breakdown.to_string())

**What to notice in the printout above:**

- **Train and test counts add to the AtGenExpress total.** If they don't, something dropped silently during the split — re-check that `atgen_mask` matched what you expected.
- **Every class appears in both train *and* test.** Stratification guarantees this provided each class has at least two samples in the full pool. If a class shows zero in either column, that class is too small to support a 75/25 split and Section 3 will not be able to learn or evaluate it. Stop and reconsider the precommitted scope in that case.
- **The smallest class's `test_n` is your binding constraint on per-class recall.** A class with `test_n = 3` produces recall values of 0%, 33%, 67%, or 100% — no other outcomes are possible. Section 4's per-class precision and recall table needs to be read with this granularity in mind. If a small class's recall is reported as 67%, that's two of three test samples correctly classified, which is not statistically distinguishable from 33% or 100% on three trials.

The split is now in memory as `X_train`, `X_test`, `y_train`, `y_test`. These flow into Section 3 (training) and Section 4 (evaluation) — and `X_test` plus `y_test` plus their metadata become `atgenexpress_test_split.parquet` in Section 6.

Section 3 trains the classifier on `X_train` and `y_train`. The model architecture and hyperparameter choices live there.

### 3) Train the multi-class stress classifier

This section trains a multi-class logistic regression classifier on `X_train` and `y_train`. Logistic regression is the textbook starting point for high-dimensional genomic classification: it scales to ~22,000 features without difficulty, it produces per-class probability estimates (useful in Section 4 and again in Notebook 03), and its coefficients are interpretable per gene (useful if you want to extend the analysis later to ask which genes drove a particular prediction).

**The model is L2-regularized.** With ~22,000 gene features and on the order of a thousand training samples, the model has many more parameters than data points. Without regularization, the fit would memorize the training set and generalize poorly — the classic high-dimensional overfitting failure mode. L2 regularization (a penalty on the sum of squared coefficients) shrinks every coefficient toward zero, which constrains the model to use *combinations* of genes rather than letting any single gene dominate.

**The regularization strength is the one hyperparameter you set yourself.** Sklearn parameterizes regularization strength as `C`, which is the *inverse* of the penalty — lower `C` means stronger regularization. The practice cell below asks you to pick a value and defend it.

**Class weighting is set to balanced.** Because the stress classes are imbalanced (Section 2's per-class breakdown made that visible), `class_weight="balanced"` reweights each class by the inverse of its frequency, so the model doesn't simply learn to predict the majority class. Some residual imbalance effects will still surface in Section 4's per-class metrics — that's expected and informative, not a failure.

**Features get standardized first.** Logistic regression with L2 is scale-sensitive: a gene whose values run 0–100 contributes more to the penalty than a gene whose values run 0–1, even if their biological meaning is identical. The next cell standardizes every gene column to zero mean and unit variance using statistics learned from the training set only — the test set gets transformed with those same statistics, never re-fit on its own. Re-fitting on the test set would leak test-set information into training and inflate the Section 4 numbers.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Fit the scaler on TRAINING data only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to the test set — never re-fit.
X_test_scaled = scaler.transform(X_test)

# Sanity confirmation.
print(f"X_train_scaled: shape {X_train_scaled.shape}, "
      f"mean ≈ {X_train_scaled.mean():.4f}, std ≈ {X_train_scaled.std():.4f}")
print(f"X_test_scaled:  shape {X_test_scaled.shape}, "
      f"mean ≈ {X_test_scaled.mean():.4f}, std ≈ {X_test_scaled.std():.4f}")

# X_train_scaled should be very close to mean 0, std 1 (fit on this data).
# X_test_scaled will have mean and std close to but not exactly 0 and 1 —
# that's correct, because the test set was transformed with the *train* set's
# statistics, not its own. If X_test_scaled also had exactly mean 0 / std 1,
# that would mean the scaler was re-fit on test data, which is the leak we're
# preventing.

**Practice 3.1 — pick `C`.**

`C` controls the trade-off between fitting the training data well (high `C`, weak regularization) and keeping coefficients small (low `C`, strong regularization). Standard values to consider:

| `C` value | Effect |
|---|---|
| `0.01` | Very strong regularization. Most gene coefficients pushed close to zero. Underfits if too aggressive — model may fail to learn distinctions between similar stress classes. |
| `0.1` | Strong regularization. A reasonable default starting point when features vastly outnumber samples, as here. |
| `1.0` | Sklearn's default. Moderate regularization. Reasonable when features and samples are comparable in number; arguably too weak here. |
| `10.0` | Weak regularization. Approaches unregularized logistic regression. Will likely overfit at this feature-to-sample ratio. |

**Pick one and write a one-paragraph defense in the markdown cell that follows the practice cell.** Your defense should reference (a) the feature-to-sample ratio you're working with (gene count from Section 1's print vs `len(X_train)` from Section 2's print) and (b) why your chosen `C` is appropriate at that ratio.

There is no single "right" answer in the range `0.01` to `1.0`. The grading concern is whether your reasoning matches your choice, not whether your choice matches a key.

**For the curious:** the principled way to choose `C` is cross-validation on the training set — sklearn's `LogisticRegressionCV` does this automatically. We're not doing that here, because the point of this exercise is for you to defend a manual choice based on principle. If you later want to compare your manual `C` against the cross-validated optimum, that's a great extension.

In [ ]:
# Practice 3.1 — your code here.
#
# Goal: pick a value of C and assign it to the variable below. Then write
# your one-paragraph defense in the markdown cell that follows.
#
# Hint: see the table in the cell above for standard candidates. There is
# no single right answer in the 0.01–1.0 range; the defense matters more
# than the specific number.

C_VALUE = None  # Replace None with your chosen value (a float).

# Sanity check — the assertion below stops execution until you make a choice.
assert C_VALUE is not None, "Practice 3.1 — set C_VALUE before continuing."
assert isinstance(C_VALUE, (int, float)) and C_VALUE > 0, \
    "C_VALUE must be a positive number."

print(f"Using C = {C_VALUE}")

*(Write your one-paragraph defense of your `C_VALUE` here. Replace this italicized text with your paragraph. Reference the feature-to-sample ratio and why your chosen C is appropriate at that ratio.)*

In [ ]:
from sklearn.linear_model import LogisticRegression

# Multi-class L2 logistic regression.
#   penalty="l2"           — L2 regularization (the discussion in 3.B).
#   C=C_VALUE              — your choice from Practice 3.1.
#   class_weight="balanced" — reweight each class by inverse frequency.
#   solver="lbfgs"         — efficient for L2 + multi-class.
#   max_iter=2000          — raised from the default 100 to give the solver
#                            room to converge at high dimensionality.
#   multi_class="multinomial" — true multi-class objective, not one-vs-rest.
#                               (Default in current sklearn, set explicitly
#                               here for clarity.)
#   random_state=RANDOM_STATE — reproducibility.
classifier = LogisticRegression(
    penalty="l2",
    C=C_VALUE,
    class_weight="balanced",
    solver="lbfgs",
    max_iter=2000,
    multi_class="multinomial",
    random_state=RANDOM_STATE,
)

classifier.fit(X_train_scaled, y_train)

# Confirm the fit completed cleanly.
print(f"Classifier trained.")
print(f"  Classes learned:   {list(classifier.classes_)}")
print(f"  Coefficient shape: {classifier.coef_.shape}  (n_classes × n_features)")
print(f"  Iterations used:   {classifier.n_iter_[0]}  (max_iter = {classifier.max_iter})")

if classifier.n_iter_[0] >= classifier.max_iter:
    print()
    print("WARNING: the solver hit max_iter without converging.")
    print("This usually means C is very large (weak regularization) at this")
    print("feature-to-sample ratio. Consider a smaller C, or raise max_iter.")

**What's now in memory:**

- `scaler` — fit on `X_train`. Notebook 03 will need to re-create this from the saved classifier, so Section 6 packages them together.
- `classifier` — the trained L2 logistic regression model.
- `X_train_scaled`, `X_test_scaled` — features ready for prediction.

Section 4 generates predictions on `X_test_scaled` and computes the within-AtGenExpress evaluation metrics (overall accuracy, per-class precision and recall). Those numbers feed the accuracy gate in Section 5.

### 4) Evaluate on the held-out AtGenExpress split

This section runs three evaluations on the held-out AtGenExpress test set, each answering a different question:

- **Overall accuracy** — does the classifier get it right more often than chance overall? Section 5's accuracy gate uses this as its headline number.
- **Per-class precision, recall, F1, and support** — does the classifier work across all stress classes, or only on the well-sampled ones? Imbalanced training data does not automatically produce a classifier that's equally good at every class.
- **Confusion matrix** — when the classifier is wrong, what does it get wrong about? Off-diagonal entries reveal which class pairs the model has trouble separating.

All three are computed in this section and held in memory. The accuracy gate in Section 5 reads them; the closeout in Section 6 saves them to disk.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

# Generate predictions on the held-out test set.
# classifier.predict returns class labels in the same order as the input rows,
# so y_pred is positionally aligned with y_test (and with X_test).
y_pred = classifier.predict(X_test_scaled)

# Overall accuracy — single number, fraction of test samples classified correctly.
overall_accuracy = accuracy_score(y_test, y_pred)

print(f"Predictions generated for {len(y_pred):,} test samples.")
print(f"Overall accuracy: {overall_accuracy:.3f}  ({overall_accuracy:.1%})")

In [ ]:
# Per-class precision, recall, F1, and support.
# zero_division=0 silences the warning that fires when a class has zero
# predictions (precision undefined) or zero true samples (recall undefined).
# The support column lets the reader tell "truly zero" from "undefined".
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred,
    labels=classifier.classes_,
    zero_division=0,
)

# Macro average — unweighted mean across classes. Every class counts equally,
# regardless of how many samples it has.
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0,
)

# Weighted average — mean across classes weighted by support. Closer to
# overall accuracy when one class dominates.
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted", zero_division=0,
)

# Build the per-class rows, sorted by support descending so the largest
# classes appear first and the smallest (noisiest) sit just above the averages.
per_class_rows = pd.DataFrame({
    "class":     classifier.classes_,
    "precision": precision,
    "recall":    recall,
    "f1":        f1,
    "support":   support,
}).sort_values("support", ascending=False)

# Append macro and weighted average rows. Support for both = total test N.
total_support = int(support.sum())
avg_rows = pd.DataFrame({
    "class":     ["macro_avg",  "weighted_avg"],
    "precision": [prec_macro,   prec_w],
    "recall":    [rec_macro,    rec_w],
    "f1":        [f1_macro,     f1_w],
    "support":   [total_support, total_support],
})

metrics_df = pd.concat([per_class_rows, avg_rows], ignore_index=True)

# Print with 3-decimal float formatting for readability.
print("Per-class metrics on the held-out AtGenExpress test set:")
print()
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

**What to notice in the metrics table:**

- **Overall accuracy** (from the cell above) is the headline number, but it's the most easily misleading one. A classifier that always predicts the majority class will look respectable if the majority class is large enough.
- **Per-class precision** asks: when the classifier *says* `cold`, how often is it right? Low precision on a class means the model is over-predicting that class — false positives.
- **Per-class recall** asks: when the truth *is* `cold`, how often does the classifier say so? Low recall on a class means the model is missing samples of that class — false negatives.
- **F1** is the harmonic mean of precision and recall. A class with high F1 is doing well on both. A class with low F1 has a problem with at least one.
- **Support** is the test-set sample count per class. The granularity point from Section 2 lands here: a class with support of 3 produces precision and recall values that are extremely noisy — only a handful of distinct values are reachable. Read small-support metrics with that caveat in mind.
- **Macro average** treats every class equally. **Weighted average** weights each class by its support, which makes it closer to overall accuracy. The gap between the two is a direct measure of how much class imbalance is affecting the headline numbers — a large gap means the small classes are doing systematically worse than the large ones.

If a class shows `precision = 0.000` *and* `support > 0`, that's a real zero — the classifier never correctly identified that class. If a class shows `precision = 0.000` *and* the model never predicted that class at all, precision is technically undefined and the table displays zero by convention (the `zero_division=0` in the code above). Either situation is informative for Section 5's gate decision.

In [ ]:
# Confusion matrix on the held-out test set.
# Rows = true class, columns = predicted class. classifier.classes_ sets the
# label order so rows and columns are consistent and labeled.
cm = confusion_matrix(y_test, y_pred, labels=classifier.classes_)

confusion_df = pd.DataFrame(
    cm,
    index=pd.Index(classifier.classes_,   name="true"),
    columns=pd.Index(classifier.classes_, name="predicted"),
)

print("Confusion matrix (rows = true class, columns = predicted class):")
print()
print(confusion_df.to_string())

**What to notice in the confusion matrix:**

- **The diagonal shows correct predictions.** Everything off the diagonal is a confusion.
- **Read a row to assess recall.** Reading along the `cold` row tells you where the actual `cold` samples ended up. Most should land in the `cold` column; the rest are false negatives, scattered across whatever other classes got predicted instead.
- **Read a column to assess precision.** Reading down the `cold` column tells you which true classes got predicted as `cold`. Everything outside the `cold` row in that column is a false positive for `cold`.
- **Look for confusable pairs.** If a particular off-diagonal entry is consistently large in *both* directions — high `cold → control` and high `control → cold`, for instance — that's a class pair the model has trouble separating in either direction. These are the most interesting cases to think about: what biology might make those two stress responses look alike at the transcriptional level?

Notebook 03 will compare this confusion matrix against the cross-dataset confusion matrix it produces. Pairs that confuse here will probably confuse there too, since the transfer task is strictly harder. The more diagnostic finding is the reverse — pairs that *don't* confuse here but *do* confuse in transfer. Those reveal class distinctions that depend on platform-specific features rather than on stress biology.

Section 5 takes the headline accuracy from Cell 4.C and the per-class numbers from `metrics_df` and runs the accuracy gate: is the within-platform classifier good enough that running the transfer test in Notebook 03 is worth the time?

### 5) Accuracy gate: is the within-platform classifier good enough that transfer is worth running? (the explicit "stop here if not" check)

You have a trained classifier and a held-out test set's worth of metrics. Before spending Week 4 on a cross-dataset transfer test, you need to be honest about whether the within-platform classifier is good enough that the transfer experiment will actually mean something.

This section asks you to score the classifier against three explicit criteria and assemble a verdict. The logic is the same Pattern D gate you ran at the end of Notebook 01:

- **Each criterion gets one of three verdicts: `pass`, `partial`, or `fail`.** The criteria and their thresholds are defined in the next cell.
- **The overall verdict rolls up from the three criterion verdicts** — `fail` if any criterion fails; `partial` if any is partial (and none fail); `pass` only if all three pass.
- **If the overall verdict is `fail`, the cell raises an error and refuses to write `classifier_summary.json`.** Notebook 03's transfer test is not meaningful on a broken within-platform baseline. Return to Section 3, reconsider your `C` choice, re-evaluate, and re-run the gate.
- **If the overall verdict is `partial`, the cell writes the file with a prominent warning.** You can proceed to Notebook 03, but document the trade-off in your Week 3 reflection.
- **If the overall verdict is `pass`, the cell writes the file cleanly and you proceed.**

Your job in the code cell below is to fill in a verdict and a rationale for each criterion, plus an overall rationale. The code auto-computes the diagnostic values and the rollup verdict; you supply the judgment and the defense.

**Three criteria.**

| Criterion | `pass` | `partial` | `fail` |
|---|---|---|---|
| **1. Overall accuracy.** Single-number headline — fraction of held-out AtGenExpress test samples classified correctly. | `≥ 0.70` | `0.50 ≤ accuracy < 0.70` | `< 0.50` |
| **2. Minimum per-class recall.** Lowest recall across all classes in the per-class table. Guards against "classifier ignores the small classes." | `≥ 0.50` for every class | at least one class in `[0.25, 0.50)`, none below `0.25` | any class below `0.25` |
| **3. Macro vs weighted F1 gap.** `weighted_f1 - macro_f1`. Measures how much class imbalance is degrading small-class performance. | `gap ≤ 0.10` | `0.10 < gap ≤ 0.20` | `gap > 0.20` |

**Why these three.** Criterion 1 sets a floor on whether the classifier learned anything useful at all — well above multi-class chance (`1/n_classes`, roughly `0.125` for eight stress classes). Criterion 2 guards against the failure mode where the classifier is "good on average" only because it does well on the large classes and ignores the small ones — the per-class table in Section 4 already exposes the diagnostic. Criterion 3 puts a number on whether the macro and weighted averages tell consistent stories; a large gap means the headline weighted figure is hiding small-class problems.

**Why these thresholds.** Round numbers, defensible at the order-of-magnitude level. They are not magic — a student with a strong reason to deviate from a stated verdict (for example, accuracy of `0.69` rounds to `partial` by the table but is functionally indistinguishable from `pass`) should write that reasoning into the rationale and use it to justify a verdict close to the line. The criteria are guides, not laws.

**What gets written.** On a successful gate (`pass` or `partial`), the cell writes `classifier_summary.json` to `OUTPUT_DIR`. The file holds the gate verdict, the diagnostic values, the model architecture, the hyperparameters, and the train/test split metadata — everything the Week 3 draft presentation needs.

In [ ]:
# Thresholds — defined as constants so the display and the markdown criteria
# table reference the same numbers.
ACCURACY_PASS    = 0.70
ACCURACY_PARTIAL = 0.50
MIN_RECALL_PASS    = 0.50
MIN_RECALL_PARTIAL = 0.25
F1_GAP_PASS    = 0.10
F1_GAP_PARTIAL = 0.20


# ============================================================
# Auto-computed diagnostic values
# ============================================================

# overall_accuracy is already in memory from Section 4.

# Minimum per-class recall (and which class).
per_class = metrics_df[~metrics_df["class"].isin(["macro_avg", "weighted_avg"])]
min_recall_row   = per_class.loc[per_class["recall"].idxmin()]
min_class_recall = float(min_recall_row["recall"])
min_recall_class = str(min_recall_row["class"])

# Macro and weighted F1, and their gap.
macro_f1    = float(metrics_df.loc[metrics_df["class"] == "macro_avg",    "f1"].iloc[0])
weighted_f1 = float(metrics_df.loc[metrics_df["class"] == "weighted_avg", "f1"].iloc[0])
f1_gap      = weighted_f1 - macro_f1

print("Diagnostic values for your verdict:")
print()
print(f"  Criterion 1 — Overall accuracy:    {overall_accuracy:.3f}")
print(f"                Thresholds:          pass ≥ {ACCURACY_PASS}, "
      f"partial ≥ {ACCURACY_PARTIAL}, fail < {ACCURACY_PARTIAL}")
print()
print(f"  Criterion 2 — Min per-class recall: {min_class_recall:.3f}  "
      f"(class: {min_recall_class!r})")
print(f"                Thresholds:          pass ≥ {MIN_RECALL_PASS}, "
      f"partial ≥ {MIN_RECALL_PARTIAL}, fail < {MIN_RECALL_PARTIAL}")
print()
print(f"  Criterion 3 — F1 gap:               {f1_gap:.3f}  "
      f"(weighted {weighted_f1:.3f} − macro {macro_f1:.3f})")
print(f"                Thresholds:          pass ≤ {F1_GAP_PASS}, "
      f"partial ≤ {F1_GAP_PARTIAL}, fail > {F1_GAP_PARTIAL}")
print()


# ============================================================
# Practice 5.1 — your verdicts and rationales.
# ============================================================
#
# For each criterion, replace "PLACEHOLDER" with one of "pass", "partial",
# or "fail", and replace the rationale string with a defense of your verdict
# (at least one sentence; the validation step below enforces this).
#
# The "overall" entry's verdict is computed by the code below — you only
# fill in its rationale, which should defend the rollup judgment as a whole
# (why you think this is or isn't worth proceeding to Notebook 03).

gate = {
    "overall_accuracy": {
        "verdict":   "PLACEHOLDER",
        "rationale": "PLACEHOLDER — write your defense of this criterion's verdict.",
    },
    "min_class_recall": {
        "verdict":   "PLACEHOLDER",
        "rationale": "PLACEHOLDER — write your defense of this criterion's verdict.",
    },
    "f1_gap": {
        "verdict":   "PLACEHOLDER",
        "rationale": "PLACEHOLDER — write your defense of this criterion's verdict.",
    },
    "overall": {
        "verdict":   None,  # computed below — leave as None.
        "rationale": "PLACEHOLDER — write your overall defense of whether "
                     "to proceed to Notebook 03.",
    },
}


# ============================================================
# Structural validation
# ============================================================

ALLOWED_VERDICTS = {"pass", "partial", "fail"}
PLACEHOLDER_MARKER = "PLACEHOLDER"
MIN_RATIONALE_CHARS = 30

per_criterion = ["overall_accuracy", "min_class_recall", "f1_gap"]

for name in per_criterion:
    entry = gate[name]

    if entry["verdict"] not in ALLOWED_VERDICTS:
        raise ValueError(
            f"gate['{name}']['verdict'] = {entry['verdict']!r}. "
            f"Replace with one of {sorted(ALLOWED_VERDICTS)} based on the diagnostic "
            f"values printed above and the thresholds in the criteria table."
        )

    rationale = entry["rationale"].strip()
    if PLACEHOLDER_MARKER in rationale:
        raise ValueError(
            f"gate['{name}']['rationale'] still contains the placeholder text. "
            f"Replace it with your own defense of the {name!r} verdict."
        )
    if len(rationale) < MIN_RATIONALE_CHARS:
        raise ValueError(
            f"gate['{name}']['rationale'] is too short ({len(rationale)} chars). "
            f"Write at least one sentence (~{MIN_RATIONALE_CHARS} characters) "
            f"defending your verdict."
        )


# ============================================================
# Rollup — compute the overall verdict from per-criterion verdicts.
# ============================================================

verdicts = [gate[name]["verdict"] for name in per_criterion]
if "fail" in verdicts:
    overall_verdict = "fail"
elif "partial" in verdicts:
    overall_verdict = "partial"
else:
    overall_verdict = "pass"

gate["overall"]["verdict"] = overall_verdict

# Validate the overall rationale (same checks as per-criterion).
overall_rationale = gate["overall"]["rationale"].strip()
if PLACEHOLDER_MARKER in overall_rationale:
    raise ValueError(
        "gate['overall']['rationale'] still contains the placeholder text. "
        "Replace it with your defense of whether to proceed to Notebook 03."
    )
if len(overall_rationale) < MIN_RATIONALE_CHARS:
    raise ValueError(
        f"gate['overall']['rationale'] is too short ({len(overall_rationale)} chars). "
        f"Write at least one sentence (~{MIN_RATIONALE_CHARS} characters)."
    )


# ============================================================
# Refuse on fail — no file write.
# ============================================================

if overall_verdict == "fail":
    raise RuntimeError(
        "Overall gate verdict is 'fail'. classifier_summary.json will not be written. "
        "Return to Section 3, reconsider your C choice (or revisit the integration step "
        "in Notebook 01), re-evaluate, and re-run this cell."
    )


# ============================================================
# Assemble the full classifier_summary.json payload.
# ============================================================

summary = {
    "gate": gate,
    "diagnostics": {
        "overall_accuracy":  float(overall_accuracy),
        "min_class_recall":  min_class_recall,
        "min_recall_class":  min_recall_class,
        "macro_f1":          macro_f1,
        "weighted_f1":       weighted_f1,
        "f1_gap":            float(f1_gap),
    },
    "architecture": {
        "model":         "LogisticRegression",
        "penalty":       "l2",
        "solver":        "lbfgs",
        "multi_class":   "multinomial",
        "class_weight":  "balanced",
    },
    "hyperparameters": {
        "C":            float(C_VALUE),
        "max_iter":     int(classifier.max_iter),
        "random_state": int(RANDOM_STATE),
    },
    "split": {
        "n_train":       int(len(X_train)),
        "n_test":        int(len(X_test)),
        "test_fraction": float(TEST_FRACTION),
        "stratified_by": "stress_label",
    },
}


# ============================================================
# Write and report.
# ============================================================

summary_path = OUTPUT_DIR / "classifier_summary.json"
with summary_path.open("w") as f:
    json.dump(summary, f, indent=2)

print(f"Overall gate verdict: {overall_verdict!r}")
print(f"classifier_summary.json written to {summary_path}")

if overall_verdict == "partial":
    print()
    print("=" * 60)
    print("WARNING: at least one criterion came back partial.")
    print("Proceeding to Notebook 03 is permitted, but document")
    print("the trade-off in your Week 3 reflection.")
    print("=" * 60)

### 6) Closeout: save `classifier.pkl`, the test split, the per-class metrics, and the confusion matrix; submit Week 3 draft presentation

Section 5 already wrote `classifier_summary.json` to `OUTPUT_DIR`. This section writes the remaining four deliverables:

- **`classifier.pkl`** — a dict `{"scaler": scaler, "classifier": classifier}`, pickled via joblib. Bundling the scaler with the classifier means Notebook 03 can load both as a single artifact and apply them consistently to the Wang rows. The dict structure makes the contract explicit — Notebook 03 reads `bundle["scaler"]` and `bundle["classifier"]` by key, not by position.
- **`atgenexpress_test_split.parquet`** — the held-out AtGenExpress test rows pulled out of `integrated_matrix.parquet` by sample ID, with the same column layout (gene columns plus `batch`, `stress_label`, `ood`). Notebook 03 uses this when it needs to refer to the same samples this notebook tested on, without re-doing the train/test split.
- **`classifier_metrics.parquet`** — the per-class metrics table (`metrics_df`) from Section 4.
- **`classifier_confusion.parquet`** — the confusion matrix (`confusion_df`) from Section 4.

The code cell below assembles the test-split DataFrame from in-memory objects, writes all four files, and runs a brief on-disk sanity check before reporting success.

In [ ]:
import joblib

# ----- Assemble the test split DataFrame -----
# atgen_df is the AtGenExpress portion of integrated_matrix (built in Section 2),
# indexed by sample ID. X_test.index carries the same sample IDs that came out of
# the stratified train_test_split. Re-indexing atgen_df by X_test.index recovers
# the full row (gene values + batch + stress_label + ood) for each held-out sample.
test_split_df = atgen_df.loc[X_test.index].copy()

# Sanity: the assembled DataFrame should have the same number of rows as X_test
# and the same columns as integrated_matrix.
assert len(test_split_df) == len(X_test), (
    f"Test split row count mismatch: assembled {len(test_split_df)}, "
    f"expected {len(X_test)}."
)
assert list(test_split_df.columns) == list(integrated_matrix.columns), (
    "Test split columns do not match integrated_matrix columns. "
    "atgen_df was built from integrated_matrix in Section 2; this should not happen."
)
assert (test_split_df["stress_label"].values == y_test.values).all(), (
    "stress_label column of the test split does not match y_test. "
    "Re-check the indexing in Section 2."
)


# ----- Define paths -----
classifier_path = OUTPUT_DIR / "classifier.pkl"
test_split_path = OUTPUT_DIR / "atgenexpress_test_split.parquet"
metrics_path    = OUTPUT_DIR / "classifier_metrics.parquet"
confusion_path  = OUTPUT_DIR / "classifier_confusion.parquet"


# ----- Write the four files -----

# classifier.pkl: dict bundling scaler + classifier, joblib-serialized.
# joblib is sklearn's recommended serializer for sklearn objects — more
# efficient than pickle for objects with large numpy arrays (the
# coefficient matrix here is ~8 classes × ~22,000 genes).
bundle = {"scaler": scaler, "classifier": classifier}
joblib.dump(bundle, classifier_path)

# Three parquets.
test_split_df.to_parquet(test_split_path)
metrics_df.to_parquet(metrics_path)
confusion_df.to_parquet(confusion_path)


# ----- Confirm on disk -----
summary_path = OUTPUT_DIR / "classifier_summary.json"  # written by Section 5

written_files = [
    ("classifier.pkl",                   classifier_path),
    ("atgenexpress_test_split.parquet",  test_split_path),
    ("classifier_metrics.parquet",       metrics_path),
    ("classifier_confusion.parquet",     confusion_path),
    ("classifier_summary.json",          summary_path),  # from Section 5
]

print("All Notebook 02 deliverables on disk:")
print()
for name, path in written_files:
    if not path.exists():
        raise FileNotFoundError(
            f"{name} was expected at {path} but is not present. "
            "Re-run this cell (or Section 5, for classifier_summary.json)."
        )
    size_kb = path.stat().st_size / 1024
    print(f"  {name:<40s}  {size_kb:>10,.1f} KB")

**Week 3 deliverable.** Submit your Week 3 draft presentation per the procedure on the R1-Q4 page in Notion. The presentation draws on:

- `classifier_summary.json` — model architecture, hyperparameters (including the `C` you defended in Section 3), train/test split sizes, headline accuracy, and the gate verdict from Section 5.
- `classifier_metrics.parquet` — per-class precision, recall, F1, and support, plus macro and weighted averages.
- `classifier_confusion.parquet` — the diagnostic for which class pairs the classifier struggles to separate.

If Section 5's gate verdict was `"partial"`, your presentation must explicitly address the trade-off — name which criterion came back partial, the diagnostic value, your rationale for proceeding anyway, and what you'll watch for in Notebook 03 as a result.

**What Notebook 03 will expect.** Notebook 03 (Week 4, the cross-dataset transfer test) loads three of the files this notebook wrote:

```python
bundle = joblib.load(OUTPUT_DIR / "classifier.pkl")
scaler     = bundle["scaler"]
classifier = bundle["classifier"]

within_confusion = pd.read_parquet(OUTPUT_DIR / "classifier_confusion.parquet")
within_metrics   = pd.read_parquet(OUTPUT_DIR / "classifier_metrics.parquet")
```

It also re-loads `integrated_matrix.parquet` from Notebook 01 to access the Wang rows, applies the scaler to them, generates predictions with the classifier, and builds a cross-dataset confusion matrix to compare against `within_confusion`. The pairs that confuse here will probably confuse there too — the interesting finding is the reverse, the pairs that didn't confuse here but do in transfer.

Notebook 02 is done. Open `03_wang_evaluation.ipynb` when you're ready to start Week 4.